# 3.3 - City-parameterised Airbnb ETL

One notebook and module handle every city. The selected archive is copied unchanged to a Unity Catalog volume, loaded as raw strings in bronze, cleaned/typed in silver, and projected into gold model inputs. Model fitting and the analytical gold tables belong to separate deliverables.

Default namespace: `workspace.airbnb`. Outputs: `bronze_listings`, `silver_listings`, `gold_model_features`. Existing EDA, model and dashboard tables are not modified. `Santa_Clara_County` is blocked unless `allow_city_c=true` is explicitly supplied after the A/B freeze.

Runtime parameters are notebook widgets and can be overridden by a Databricks Job. An optional rerun check compares stable content signatures, excluding the changing ingestion timestamp.

In [0]:
dbutils.widgets.dropdown("city", "Los_Angeles", ["Los_Angeles", "San_Francisco", "Santa_Clara_County"])
dbutils.widgets.text("catalog", "workspace")
dbutils.widgets.text("schema", "airbnb")
dbutils.widgets.text("repo_root", "")
dbutils.widgets.text("source_dir", "")
dbutils.widgets.dropdown("allow_city_c", "false", ["false", "true"])
dbutils.widgets.dropdown("verify_rerun", "false", ["false", "true"])

## Resolve source code and configuration

`repo_root` is the Databricks folder containing `src/` and `data/`; blank enables detection from the notebook's working directory. `source_dir` defaults to that folder's `data/`, or can point to an archive directory in a volume. No EDA-table read permission is required.

In [0]:
from pathlib import Path
import importlib
import json
import sys
from pyspark.sql import functions as F

root_parameter = dbutils.widgets.get("repo_root").strip()
cwd = Path.cwd()
candidates = [Path(root_parameter)] if root_parameter else [cwd, *cwd.parents]
repo_root = next((p for p in candidates if (p / "src" / "etl_pipeline.py").is_file()), None)
if repo_root is None:
    raise FileNotFoundError(f"src/etl_pipeline.py not found from {cwd}; set repo_root to the full Databricks project path")
src_dir = str(repo_root / "src")
if src_dir not in sys.path:
    sys.path.insert(0, src_dir)
import etl_pipeline as etl
importlib.reload(etl)

config = etl.PipelineConfig(
    city=dbutils.widgets.get("city"),
    catalog=dbutils.widgets.get("catalog"),
    schema=dbutils.widgets.get("schema"),
    source_dir=dbutils.widgets.get("source_dir").strip() or str(repo_root / "data"),
    allow_city_c=dbutils.widgets.get("allow_city_c") == "true",
)
config.validate()
print(f"City: {config.city}; namespace: {config.namespace}")
print(f"Source directory: {config.source_dir}")
print("Outputs:", json.dumps(config.tables, indent=2))

City: San_Francisco; namespace: workspace.airbnb
Source directory: /Workspace/Users/oleg.hutsuliak@gmail.com/PizzaByte_group_assignment_2/data
Outputs: {
  "bronze": "workspace.airbnb.bronze_listings",
  "silver": "workspace.airbnb.silver_listings",
  "features": "workspace.airbnb.gold_model_features"
}


## Ingest, transform, validate and publish one city

`replaceWhere` limits refreshes to the selected city. A/B row-count expectations apply to the supplied archives; missing optional columns become typed nulls. IDs are parsed directly as BIGINT; invalid keys fail visibly and duplicates are resolved by scrape date plus a deterministic source-record hash. Rows without usable labels remain in silver and gold for prediction.

In [0]:
summary = etl.run_pipeline(spark, config)
print(json.dumps(summary, indent=2, default=str))
print("SELECTED_CITY_CHECKS_PASSED")

{
  "city": "San_Francisco",
  "source_sha256": "ed20a45c3281bdae3f89265132a9eef83b0b276b6a54bcb7a3bc1a6f98af0677",
  "staged_file": "/Volumes/workspace/airbnb/raw/San_Francisco/ed20a45c3281bdae3f89265132a9eef83b0b276b6a54bcb7a3bc1a6f98af0677/listings_San_Francisco.csv.gz",
  "bronze_rows": 7405,
  "duplicate_rows_removed": 0,
  "rows": 7405,
  "distinct_ids": 7405,
  "valid_prices": 5676,
  "no_price": 1714,
  "placeholders": 15,
  "stale": 1682,
  "amenities_parse_failures": 0,
  "first_scraped": "2026-09-14",
  "last_scraped": "2026-09-14",
  "tables": {
    "bronze": "workspace.airbnb.bronze_listings",
    "silver": "workspace.airbnb.silver_listings",
    "features": "workspace.airbnb.gold_model_features"
  }
}
SELECTED_CITY_CHECKS_PASSED


## Label eligibility and stay types

One row per eligibility class and stay type for the selected city. `price_class` uses fixed business rules (`valid`, `no_price`, `placeholder`), not EDA-wide statistical fences. Eligibility matches `price_model.valid_price_mask`; no imputer, encoder or model is fitted here.

In [0]:
silver = spark.table(config.tables["silver"]).filter(F.col("city") == config.city)
display(silver.groupBy("price_class", "stay_type").agg(F.count("*").alias("listings"))
        .orderBy("price_class", "stay_type"))
display(silver.select("city", "id", "price_text", "price", "price_valid", "bathrooms_text",
                      "bathrooms_num", "bathroom_shared", "n_amenities", "stay_type").limit(5))

price_class,stay_type,listings
no_price,monthly,901
no_price,short,813
placeholder,monthly,4
placeholder,short,11
valid,monthly,1999
valid,short,3666
valid,unknown,11


city,id,price_text,price,price_valid,bathrooms_text,bathrooms_num,bathroom_shared,n_amenities,stay_type
San_Francisco,8142,$66.87,66.87,true,4 shared baths,4.0,true,21,monthly
San_Francisco,10537,$171.00,171.0,true,1.5 shared baths,1.5,true,58,short
San_Francisco,24463,$281.00,281.0,true,1 private bath,1.0,false,51,short
San_Francisco,41055,$135.80,135.8,true,1 bath,1.0,false,49,monthly
San_Francisco,47486,null,null,false,1 bath,1.0,false,31,monthly


## Model input contract

One row per required input column with its Spark type. The raw input names match `src/price_model.py`. `price` is the target; `city` and `id` are keys/metadata. Predictors are chosen explicitly by the model, never by taking all remaining columns. Learned preprocessing remains inside the training split.

In [0]:
features = spark.table(config.tables["features"]).filter(F.col("city") == config.city)
types = dict(features.dtypes)
assert set(etl.MODEL_INPUT_COLUMNS).issubset(types), "Model input column missing"
assert not any(c.startswith("price_quote_") for c in features.columns), "Quote leakage in feature projection"
assert "estimated_revenue_l365d" not in features.columns, "Revenue leakage in feature projection"
display(spark.createDataFrame([(c, types[c]) for c in etl.MODEL_INPUT_COLUMNS], "column string, spark_type string"))

column,spark_type
city,string
id,bigint
room_type,string
property_type,string
price,double
accommodates,bigint
bedrooms,double
beds,double
bathrooms_num,double
bathroom_shared,boolean


## Optional idempotence check

When `verify_rerun=true`, the same city runs a second time. Stable row/key counts and content hashes across all currently published A/B rows must be identical in bronze, silver and the feature table. The deliberately refreshed `_ingested_at` timestamp is excluded. This checks both repeatability and preservation of the other A/B city; it does not read C.

In [0]:
if dbutils.widgets.get("verify_rerun") == "true":
    before = {k: etl.table_signature(spark, t) for k, t in config.tables.items()}
    repeated = etl.run_pipeline(spark, config)
    after = {k: etl.table_signature(spark, t) for k, t in config.tables.items()}
    assert before == after, "Rerun changed stable A/B contents or duplicated/dropped rows"
    print("IDEMPOTENCE_CHECK_PASSED: stable A/B contents and row counts unchanged")
    print(json.dumps(after, indent=2))
else:
    print("Rerun check not requested for this execution")

IDEMPOTENCE_CHECK_PASSED: stable A/B contents and row counts unchanged
{
  "bronze": [
    [
      "Los_Angeles",
      43781,
      43781,
      "1485119811292898236806"
    ],
    [
      "San_Francisco",
      7405,
      7405,
      "518120725336747521278"
    ]
  ],
  "silver": [
    [
      "Los_Angeles",
      43781,
      43781,
      "734226607156776864895"
    ],
    [
      "San_Francisco",
      7405,
      7405,
      "-43246795442875150413"
    ]
  ],
  "features": [
    [
      "Los_Angeles",
      43781,
      43781,
      "157337460048968504645"
    ],
    [
      "San_Francisco",
      7405,
      7405,
      "398037067474526323357"
    ]
  ]
}


## Published A/B tables

One row per layer and currently loaded A/B city. After both city runs, each layer should contain 43,781 LA and 7,405 SF rows, with unique listing keys. A first run for one city does not require the other city to have been loaded already. No prediction or analytical gold table is overwritten by this pipeline.

In [0]:
published = []
for layer, table_name in config.tables.items():
    counts = (spark.table(table_name).filter(F.col("city").isin(list(etl.TRAINING_CITIES)))
              .groupBy("city").agg(F.count("*").alias("rows"), F.countDistinct("id").alias("ids")))
    for row in counts.collect():
        assert row.rows == row.ids, f"Duplicate IDs in {table_name}"
        assert row.rows == etl.EXPECTED_ROWS[row.city], f"Unexpected supplied-file row count in {table_name}"
        published.append((layer, table_name, row.city, row.rows, row.ids))
display(spark.createDataFrame(published, "layer string, table_name string, city string, rows long, distinct_ids long"))
display(spark.table(config.tables["silver"]).filter(F.col("city").isin(list(etl.TRAINING_CITIES)))
    .groupBy("city").agg(F.count("*").alias("listings"),
        F.sum(F.col("price_valid").cast("long")).alias("valid_prices"),
        F.sum((F.col("price_class") == "no_price").cast("long")).alias("no_price"),
        F.sum((F.col("price_class") == "placeholder").cast("long")).alias("placeholders"))
    .orderBy("city"))
print("PIPELINE_CHECKS_PASSED")

layer,table_name,city,rows,distinct_ids
bronze,workspace.airbnb.bronze_listings,San_Francisco,7405,7405
bronze,workspace.airbnb.bronze_listings,Los_Angeles,43781,43781
silver,workspace.airbnb.silver_listings,San_Francisco,7405,7405
silver,workspace.airbnb.silver_listings,Los_Angeles,43781,43781
features,workspace.airbnb.gold_model_features,San_Francisco,7405,7405
features,workspace.airbnb.gold_model_features,Los_Angeles,43781,43781


city,listings,valid_prices,no_price,placeholders
Los_Angeles,43781,36676,7094,11
San_Francisco,7405,5676,1714,15


PIPELINE_CHECKS_PASSED
